In [ ]:
import os
import pyModeS as pms
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

def encrypt_AESGCM(msg: list[str], counter: int, model) -> tuple[list, int]:
    """
    Model aircraft-side secure ADS-B transmission using AES-GCM.

    The original ADS-B message is split into header (AAD: DF+CA+ICAO) and
    payload; only the payload is encrypted, while the header remains in clear.
    A nonce is constructed from random bytes and a counter, and the GCM tag
    is appended alongside the ciphertext. The CRC is recomputed over the
    modified message, assuming tag and nonce are conveyed as associated
    metadata (e.g., overlay channel). Returns the protected message and the
    incremented counter.
    """

    # Skip empty message
    if not msg or not msg[0]:
        return msg, counter

    # Compute nonce: 8 random bytes + 4-byte counter
    nonce = os.urandom(8) + counter.to_bytes(4, "big")

    # Convert ADS-B message to bytes
    msg_bytes = bytes.fromhex(msg[0])

    # Split header (AAD) and payload
    aad = msg_bytes[:4]  # DF + CA + ICAO
    payload = msg_bytes[4:]  # Remaining bytes

    # Encrypt and authenticate
    ct_full = model.encrypt(nonce, payload, aad)  # type: ignore

    # Split ciphertext and tag (last 16 bytes)
    tag = ct_full[-16:]
    ct = ct_full[:-16]

    # Recompute CRC over AAD + ciphertext
    aad_hex = aad.hex().upper()
    ct_hex = ct.hex().upper()

    msg_for_crc = aad_hex + ct_hex + "000000"
    crc_value = pms.crc(msg_for_crc, encode=True)
    crc_hex = f"{crc_value:06X}"

    full_msg = aad_hex + ct_hex + crc_hex

    # Return encrypted structure
    return [full_msg, tag, nonce], counter + 1

def AESGCM_check_nonce(nonce: bytes, cached_nonce: bytes) -> bool:
    """
    Compare the given nonce with the last stored nonce for aircraft i.
    Returns True if nonce is valid (counter > last), False otherwise.
    """

    # If no cached nonce, accept it
    if not cached_nonce:
        return True

    # Extract counter from 4 last bytes of the nonce
    counter = int.from_bytes(nonce[-4:], "big")
    cached_counter = int.from_bytes(cached_nonce[-4:], "big")

    if counter <= cached_counter:
        return False  # replay or old message

    return True

def decrypt_AESGCM(
    msg: list, cached_nonce: bytes, model
) -> tuple[list[str], bytes]:
    """
    Model receiver-side AES-GCM decryption and authentication of a protected
    ADS-B message.

    The header (AAD) is used for authentication while the payload is decrypted
    using the provided nonce and key model. Nonce freshness is enforced to
    mitigate replay; if authentication or nonce validation fails, [""] is
    returned to represent a corrupted/unauthenticated message. On success,
    the original ADS-B hex message is reconstructed and the nonce cache updated.
    """

    if len(msg) != 3:
        return [""], cached_nonce

    nonce = msg[2]

    if not AESGCM_check_nonce(nonce, cached_nonce):
        # If old message, drop it and quit
        return [""], cached_nonce
    else:
        # Else update nonce cache
        cached_nonce = nonce

    # Convert hex to bytes
    msg_bytes = bytes.fromhex(msg[0])

    # Split message in aad, tag and payload
    aad = msg_bytes[:4]
    payload = msg_bytes[4:-3]
    crc = msg_bytes[-3:]
    tag = msg[1]  # already in bytes

    # Append stored tag
    ct = payload + tag

    # Decrypt
    try:
        plaintext = model.decrypt(nonce, ct, aad)  # type:ignore
        return [(aad + plaintext + crc).hex().upper()], cached_nonce
    except Exception:
        return [""], cached_nonce
